# Smart Elevator CV — YOLOv8 Training (Colab)

## Önkoşul (sadece 1 kez yapılır — yerel makinede)

1. `cd Desktop/Capstone_deneme_ai`
2. `python -m scripts.prepare_dataset --raw <datas_klasoru> --out data/unified`
3. `python -m scripts.augment_dataset --unified data/unified --config configs/default.yaml`
4. `python -m scripts.package_for_colab`  → `Desktop/colab_upload/code.zip` + `dataset.zip` üretir
5. **İki ZIP'i de Google Drive'da `MyDrive/Capstone/` klasörüne yükleyin**
6. Bu notebook'u Drive'da bularak Colab'da açın (sağ tık → Open with → Google Colaboratory)

## Colab tarafı

Aşağıdaki hücreler:
- Drive'ı bağlar
- ZIP'leri `/content/` altına çıkarır (Drive üzerinden okumak çok yavaş)
- `requirements.txt`'yi yükler
- YOLOv8'i sıfırdan eğitir
- `best.pt`'yi Drive'a geri kopyalar

## 1. Drive'ı bağla ve ZIP'leri çıkar

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, zipfile, shutil
BASE_DRIVE = '/content/drive/MyDrive/Capstone'
WORK = '/content/work'
REPO = f'{WORK}/Capstone_deneme_ai'
DATA = f'{WORK}/data/unified'

assert os.path.exists(f'{BASE_DRIVE}/code.zip'),    f'code.zip bulunamadi: {BASE_DRIVE}/code.zip'
assert os.path.exists(f'{BASE_DRIVE}/dataset.zip'), f'dataset.zip bulunamadi: {BASE_DRIVE}/dataset.zip'

os.makedirs(REPO, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

with zipfile.ZipFile(f'{BASE_DRIVE}/code.zip') as z:
    z.extractall(REPO)
with zipfile.ZipFile(f'{BASE_DRIVE}/dataset.zip') as z:
    z.extractall(DATA)

%cd {REPO}
!ls

## 2. Bağımlılıkları kur

In [ ]:
!pip install -q -r requirements.txt

## 3. GPU & dataset doğrulaması

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi 2>/dev/null | head -15

from src.dataset.audit import audit_yolo_dataset, print_audit
from src.dataset.unify import TARGET_CLASSES
print_audit(audit_yolo_dataset(DATA), class_names=TARGET_CLASSES)

## 4. Eğitim

**Augmentation zaten yerelde yapıldı** (idempotent — burada yeniden çalıştırmaya gerek yok).

T4 GPU için varsayılan değerler:
- VARIANT = `yolov8s.pt` (dengeli)
- BATCH = 32
- EPOCHS = 100

Daha hızlı denemek için `yolov8n.pt` ve `EPOCHS=50` deneyin.

In [ ]:
from ultralytics import YOLO

VARIANT = 'yolov8s.pt'
EPOCHS = 100
BATCH = 32
IMGSZ = 640

RUNS_DIR = f'{BASE_DRIVE}/models/runs'
os.makedirs(RUNS_DIR, exist_ok=True)

model = YOLO(VARIANT)
results = model.train(
    data=f'{DATA}/data.yaml',
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    lr0=0.01,
    patience=25,
    seed=42,
    project=RUNS_DIR,
    name='elevator_v1',
    plots=True,
    save_period=10,
    device=0 if torch.cuda.is_available() else 'cpu',
)
BEST = f'{results.save_dir}/weights/best.pt'
print('best.pt:', BEST)

## 5. Test set'te değerlendirme

Test set, `Elevator.yolov8` (sizin asansör verisi) + diğer kaynakların 5%'lik dilimleridir. Modelin hiç görmediği veridir.

In [ ]:
from ultralytics import YOLO
model = YOLO(BEST)
metrics = model.val(data=f'{DATA}/data.yaml', split='test')
print('mAP50:    ', metrics.box.map50)
print('mAP50-95: ', metrics.box.map)
print('Per-class mAP50:', dict(zip(metrics.names.values(), metrics.box.maps.tolist())))

## 6. `best.pt`'yi Drive'a (ve repo'ya) kopyala

In [ ]:
import shutil
drive_dst = f'{BASE_DRIVE}/models/weights/best.pt'
repo_dst  = f'{REPO}/models/weights/best.pt'
for d in (drive_dst, repo_dst):
    os.makedirs(os.path.dirname(d), exist_ok=True)
    shutil.copy2(BEST, d)
    print('Copied:', d)

## 7. Sample inference

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import glob

test_imgs = sorted(glob.glob(f'{DATA}/test/images/*.jpg'))[:6]
model = YOLO(repo_dst)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, test_imgs):
    res = model.predict(img_path, conf=0.4, verbose=False)[0]
    ax.imshow(Image.fromarray(res.plot()[..., ::-1]))
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Sonra ne olacak?

1. **`best.pt`'yi yerele indir** (Drive ile otomatik senkronize) → `Desktop/Capstone_deneme_ai/models/weights/best.pt`
2. Yerel makinede `python -m scripts.demo --image <foto> --weights models/weights/best.pt`
3. Notebook 03 (BEV demo) ve Notebook 04 (enerji simülasyonu) yerelde çalışacak.